# Clause importance — numerical verification

The metric adapts GraphQAG (Li et al., IEEE TVCG 2026, §IV-A): the restart mass is spread
evenly over clauses and, inside each clause, evenly over its statements.

$$\\pi(v) = \\frac{1}{|C| \\cdot |V_c|} \\qquad PR^{(t+1)} = (1 - d + dD^{(t)})\\pi + d\\sum_{j \\in N_i} \\frac{PR_j^{(t)}}{\\deg(v_j)}$$

In [12]:
import collections
import json
from pathlib import Path

import networkx as nx

DAMPING = 0.85
TOLERANCE = 1e-9
MAX_ITERATIONS = 200

KG_DIR = Path("../../infra/json/kg")
DEONTIC = ("obligations", "rights", "prohibitions")

## Graph and prior

In [13]:
def load(path):
    kg = json.loads(path.read_text(encoding="utf-8"))
    statements = [dict(n, kind=k[:-1]) for k in DEONTIC for n in kg[k]]

    ids = (
        [p["id"] for p in kg["parties"]]
        + [c["id"] for c in kg["clauses"]]
        + [t["id"] for t in kg["definedTerms"]]
        + [v["id"] for v in statements]
        + [c["id"] for c in kg["conditions"]]
        + [r["id"] for r in kg["references"]]
        + [v["id"] for v in kg["values"]]
    )
    index = {node: i for i, node in enumerate(ids)}

    adjacency = [[] for _ in ids]
    for edge in kg["edges"]:
        source, target = index.get(edge["source"]), index.get(edge["target"])
        if source is None or target is None:
            continue
        adjacency[source].append(target)
        adjacency[target].append(source)

    members = collections.defaultdict(list)
    for statement in statements:
        if statement["clauseId"]:
            members[statement["clauseId"]].append(statement["id"])

    prior = [0.0] * len(ids)
    for group in members.values():
        share = 1 / (len(members) * len(group))
        for node in group:
            prior[index[node]] += share

    return kg, ids, index, adjacency, members, prior

## The implementation under test

A line-by-line replica of the TypeScript loop.

In [14]:
def pagerank(ids, adjacency, prior, start=None):
    rank = list(start) if start else prior[:]
    for iteration in range(1, MAX_ITERATIONS + 1):
        nxt = [0.0] * len(ids)
        dangling = 0.0
        for j in range(len(ids)):
            degree = len(adjacency[j])
            if degree == 0:
                dangling += rank[j]
                continue
            share = DAMPING * rank[j] / degree
            for neighbour in adjacency[j]:
                nxt[neighbour] += share
        reinjected = 1 - DAMPING + DAMPING * dangling
        delta = 0.0
        for i in range(len(ids)):
            nxt[i] += reinjected * prior[i]
            delta += abs(nxt[i] - rank[i])
        rank = nxt
        if delta < TOLERANCE:
            break
    return rank, iteration

## Reference

`dangling=personalization` matches how the implementation reinjects leaked mass.

In [15]:
def reference(ids, adjacency, prior):
    graph = nx.MultiGraph()
    graph.add_nodes_from(ids)
    for j, neighbours in enumerate(adjacency):
        for neighbour in neighbours:
            if j < neighbour:
                graph.add_edge(ids[j], ids[neighbour])
    personalization = {ids[i]: prior[i] for i in range(len(ids))}
    return nx.pagerank(
        graph,
        alpha=DAMPING,
        personalization=personalization,
        dangling=personalization,
        tol=1e-12,
        max_iter=500,
    )

## Comparison

In [16]:
def clause_scores(rank, index, members):
    return {clause: sum(rank[index[node]] for node in group) for clause, group in members.items()}


rows = []
for path in sorted(KG_DIR.glob("*.json")):
    kg, ids, index, adjacency, members, prior = load(path)
    if not members:
        continue

    rank, iterations = pagerank(ids, adjacency, prior)
    ref = reference(ids, adjacency, prior)

    settled, _ = pagerank(ids, adjacency, prior, start=rank)
    from_uniform, _ = pagerank(ids, adjacency, prior, start=[1 / len(ids)] * len(ids))

    rows.append({
        "document": path.stem[:34],
        "nodes": len(ids),
        "clauses": len(members),
        "iterations": iterations,
        "mass": sum(rank),
        "negative": sum(1 for x in rank if x < 0),
        "residual": sum(abs(a - b) for a, b in zip(rank, settled)),
        "from_uniform": max(abs(a - b) for a, b in zip(rank, from_uniform)),
        "vs_networkx": max(abs(rank[index[k]] - ref[k]) for k in ref),
    })

for row in rows:
    print(f"{row['document']:36} {row['nodes']:>5} nodes {row['iterations']:>4} iters")
    print(f"{'':36} mass {row['mass']:.12f}  negative {row['negative']}")
    print(f"{'':36} residual {row['residual']:.2e}  uniform {row['from_uniform']:.2e}")
    print(f"{'':36} vs networkx {row['vs_networkx']:.2e}")

root_BELLICUM_MILTENYI_Supply_Agre     147 nodes  132 iters
                                     mass 1.000000000000  negative 0
                                     residual 8.20e-10  uniform 1.24e-10
                                     vs networkx 4.26e-11
target_BELLICUMPHARMACEUTICALS_INC     783 nodes  116 iters
                                     mass 1.000000000000  negative 0
                                     residual 7.90e-10  uniform 1.98e-10
                                     vs networkx 1.27e-10
target_SteelVaultCorp_20081224_10-     104 nodes  132 iters
                                     mass 1.000000000000  negative 0
                                     residual 8.20e-10  uniform 1.09e-10
                                     vs networkx 5.42e-11


## Ties

A pair of clauses with an identical score is not a rounding artefact: the prior gives every
clause the same mass, and when two hold the same number of statements in an equivalent
neighbourhood, propagation has nothing to separate them with. Fewer informative edges in
the document means more ties.

In [17]:
for path in sorted(KG_DIR.glob("*.json")):
    kg, ids, index, adjacency, members, prior = load(path)
    if not members:
        continue
    rank, _ = pagerank(ids, adjacency, prior)
    heading = {c["id"]: (c["heading"] or c["ref"] or c["id"])[:32] for c in kg["clauses"]}
    ranked = sorted(clause_scores(rank, index, members).items(), key=lambda kv: -kv[1])

    tied = [
        (position, left, right)
        for position, ((left, a), (right, b)) in enumerate(zip(ranked, ranked[1:]), start=1)
        if a == b
    ]
    print(f"\n{path.stem[:44]}  {len(tied)} tied pairs of {len(ranked) - 1}")
    for position, left, right in tied:
        print(f"   {position:>4}  {heading.get(left, left):34} = {heading.get(right, right)}")


root_BELLICUM_MILTENYI_Supply_Agreement_Summ  0 tied pairs of 10

target_BELLICUMPHARMACEUTICALS_INC_05_07_201  9 tied pairs of 107
     18  Existing Intellectual Property     = Waiver of Jury Trial
     19  Waiver of Jury Trial               = Disclaimer
     71  NOTICES                            = Section 15.4
     72  Section 15.4                       = ASSIGNMENT
     87  Governing Further Actions          = Severability and Headings
     88  Severability and Headings          = Negotiated Terms
     89  Negotiated Terms                   = Counterparts
     90  Counterparts                       = Independent Contractors
     96  Product Price                      = Notification

target_SteelVaultCorp_20081224_10-K_EX-10_16  12 tied pairs of 24
      7  Operational Specifications         = Term and Termination
      8  Term and Termination               = Section 7.1
      9  Section 7.1                        = Indemnification
     10  Indemnification                    = Sect

## What the ranking adds over counting

Spearman against ordering clauses by how many statements they hold. A value close to 1
would mean the walk reproduces the count and earns nothing; the previous party-seeded
PageRank scored 0.882 there.

In [18]:
def spearman(left, right):
    ranks_left = {key: i for i, key in enumerate(sorted(left, key=lambda k: -left[k]))}
    ranks_right = {key: i for i, key in enumerate(sorted(right, key=lambda k: -right[k]))}
    n = len(ranks_left)
    squared = sum((ranks_left[key] - ranks_right[key]) ** 2 for key in ranks_left)
    return 1 - 6 * squared / (n * (n * n - 1))


for path in sorted(KG_DIR.glob("*.json")):
    kg, ids, index, adjacency, members, prior = load(path)
    if len(members) < 3:
        continue
    rank, _ = pagerank(ids, adjacency, prior)
    scores = clause_scores(rank, index, members)
    sizes = {clause: len(group) for clause, group in members.items()}
    print(f"{path.stem[:44]:46} rho = {spearman(scores, sizes):.3f}")

root_BELLICUM_MILTENYI_Supply_Agreement_Summ   rho = 0.336
target_BELLICUMPHARMACEUTICALS_INC_05_07_201   rho = 0.778
target_SteelVaultCorp_20081224_10-K_EX-10_16   rho = 0.827
